# 任务 6：最终数据导出与指标存档

## 任务描述
1. 将经过任务 1（去重）、任务 3（会话划分）处理后的干净数据导出为 `ml_final_clean.parquet`
2. 记录最终的数据总量（RowCount）与存储体积

In [ ]:
import polars as pl
import time
import os

print(f"Polars 版本：{pl.__version__}")

In [ ]:
# 配置
input_path = "data_with_session.parquet"  # 任务 3 的输出
output_path = "ml_final_clean.parquet"    # 最终输出

start_time = time.time()

## 1. 加载已处理数据

In [ ]:
df = pl.scan_parquet(input_path)
print("数据加载完成")

## 2. 查看数据结构

In [ ]:
schema = df.collect_schema()
print(f"列数：{len(schema)}")
print(f"列名：{list(schema.names())}")

## 3. 导出数据

In [ ]:
print("正在导出数据...")

# 收集并保存（使用 ZSTD 压缩）
df_final = df.collect(engine="streaming")
df_final.write_parquet(output_path, compression="zstd")

print(f"数据已保存至：{output_path}")

## 4. 统计指标

In [ ]:
# 行数
row_count = df_final.height

# 列数
col_count = len(df_final.columns)

# 文件大小
file_size_bytes = os.path.getsize(output_path)
file_size_mb = file_size_bytes / (1024 * 1024)
file_size_gb = file_size_bytes / (1024 * 1024 * 1024)

# 平均每行字节数
bytes_per_row = file_size_bytes / row_count if row_count > 0 else 0

## 5. 结果输出

In [ ]:
end_time = time.time()
total_time = end_time - start_time

print("=" * 60)
print("数据导出完成")
print("=" * 60)

print(f"""
┌─────────────────────────────────────────────────────────────┐
│                    数据导出统计                              │
├─────────────────────────────────────────────────────────────┤
│  输出文件：{output_path:<44} │
├─────────────────────────────────────────────────────────────┤
│  数据总量（RowCount）：{row_count:>15,} 行                       │
│  列数（ColumnCount）：{col_count:>15} 列                       │
├─────────────────────────────────────────────────────────────┤
│  存储体积：{file_size_mb:>14.2f} MB                              │
│           ({file_size_gb:.4f} GB)                            │
│  平均每行：{bytes_per_row:>14.2f} 字节                         │
├─────────────────────────────────────────────────────────────┤
│  压缩方式：ZSTD                                              │
│  导出耗时：{total_time:>14.2f} 秒                            │
└─────────────────────────────────────────────────────────────┘
""")

## 6. 数据预览

In [ ]:
print("=" * 60)
print("数据预览（前 5 行）")
print("=" * 60)

print(df_final.head(5))

## 7. 数据类型统计

In [ ]:
print("\n" + "-" * 60)
print("数据类型分布")
print("-" * 60)

dtype_counts = {}
for col in df_final.columns:
    dtype = str(df_final.schema[col])
    dtype_counts[dtype] = dtype_counts.get(dtype, 0) + 1

for dtype, count in sorted(dtype_counts.items()):
    print(f"  {dtype}: {count} 列")

## 8. 指标存档

In [ ]:
print("\n" + "=" * 60)
print("指标存档")
print("=" * 60)

# 创建指标记录
metrics = pl.DataFrame({
    "metric": [
        "RowCount",
        "ColumnCount", 
        "FileSize_MB",
        "FileSize_GB",
        "BytesPerRow",
        "Compression",
        "ExportTime_Sec"
    ],
    "value": [
        str(row_count),
        str(col_count),
        f"{file_size_mb:.2f}",
        f"{file_size_gb:.4f}",
        f"{bytes_per_row:.2f}",
        "zstd",
        f"{total_time:.2f}"
    ]
})

# 保存指标
metrics.write_csv("ml_final_metrics.csv")
print("指标已保存至：ml_final_metrics.csv")

# 保存列信息
col_info = pl.DataFrame({
    "column_name": df_final.columns,
    "dtype": [str(dtype) for dtype in df_final.dtypes]
})
col_info.write_csv("ml_final_columns.csv")
print("列信息已保存至：ml_final_columns.csv")